In [2]:
import pandas as pd
df = pd.read_csv(r"C:\Users\jiten\OneDrive\Desktop\python\data_set\Unconfirmed 1215.crdownload")
df.drop_duplicates(inplace=True)

#Pre-processing
#1. converting to lowercase

df['review'] = df['review'].str.lower()

import re

def remove_url(text):
    text = re.sub( r"http\S+", " " , text)
    return text
df['review'] = df['review'].apply(remove_url)

#remove Punctuations

def remove_punctuations(text):
    test = re.sub(r"[^A-za-z0-9\s]" ,"", text)
    return text

df['review'] = df['review'].apply(remove_punctuations)

#remove html

def remove_html(text):
    test = re.sub(r"<.*?>" ,"", text)
    return text

df['review'] = df['review'].apply(remove_html)

In [3]:
#removing stopwords
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word , "")
            
    return text
df['review'] = df['review'].apply(remove_stopwords)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jiten\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jiten\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jiten\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:

# stemming
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()

    stemmed_words = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df['review'] = df['review'].apply(stemming)

#encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])
y = df["sentiment"]



In [5]:

#vectorizations
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

#dataset & Data Loader
from sklearn.model_selection import train_test_split
X_train  , X_test , y_train ,y_test = train_test_split(
    X , y, test_size=0.2 , random_state=42
)

import torch
from torch.utils.data import TensorDataset , DataLoader

X_train = X_train.toarray()
X_test = X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(train_set , shuffle = True , batch_size = 64)
test_loader = DataLoader(test_set , shuffle = True , batch_size = 64)

In [ ]:


#build our RNN
import torch.nn as nn 
import torch.optim as optim

class RNN(nn.Module):
    def __init__(self ,input_size , hidden_size = 128 , num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size , num_layers , batch_first=True)

        #fully connected layer

        self.fc = nn.Linear(hidden_size ,1)

    def forward(self , x):
        
        h0 =  torch.zeros(self.num_layers , x.size(0) , self.hidden_size)
        
        out , _ = self.rnn(x , h0)
        # 1st value = hidden statenof alll the timesteps
        #2nd  value = hidden state of last timestep

        out = self.fc(out[: , -1 , :])
        return out
    input_size = X_train.shape[1]



In [11]:
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

#Training th RNN
epochs = 10
for epoch in range(epochs):
    model.train()

    for Xb , yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)

        outputs = torch.sigmoid(outputs.squeeze())
        
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"{epoch+1}/{epochs} and loss = {loss.item()}")


1/10 and loss = 0.38005971908569336
2/10 and loss = 0.33151116967201233
3/10 and loss = 0.22332212328910828
4/10 and loss = 0.2411022186279297
5/10 and loss = 0.2579089403152466
6/10 and loss = 0.3140210509300232
7/10 and loss = 0.3014461100101471
8/10 and loss = 0.32096344232559204
9/10 and loss = 0.18107204139232635
10/10 and loss = 0.1845794916152954


In [10]:

#evaluate
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb , yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)

        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        tot_vals += yb.size(0)

        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = { correct_vals/tot_vals*100}" )

accuracy = 86.00383180397297
